# Trajectoid Rolling Kinematics & SO(3) Path Tracing Recipe

This recipe combines 3 `algebrax` tools to simulate 3D Trajectoid Non-Holonomic Rolling Kinematics:

1. **2D Target Path Velocity Gradient** (`algebrax.analysis.gradient`):
   Evaluates discrete velocity components $v_x(t), v_y(t)$ along a periodic 2D figure-eight lemniscate curve $c(t)$.
2. **Non-Holonomic SO(3) Rotation Matrix Composition** (`algebrax.matrix.core.dot`):
   Integrates 3D spatial rotation matrices $R(t) \in SO(3)$ via $R_{k+1} = R_k \cdot dR_k$ for pure rolling without slipping.
3. **Contact Matrix Sparsity Audit** (`algebrax.metrics.sparsity`):
   Audits structural matrix density and spatial trajectory tracking errors.

In [ ]:
import math

from algebrax.analysis import gradient
from algebrax.matrix.core import dot
from algebrax.metrics import sparsity

## 1. Target 2D Trajectory & Velocity Field (gradient)

We generate a periodic figure-eight lemniscate path and evaluate velocity fields.

In [ ]:
n_steps = 16
dt = 2.0 * math.pi / n_steps

path_x = {t: 5.0 * math.sin(t * dt) for t in range(n_steps)}
path_y = {t: 5.0 * math.sin(t * dt) * math.cos(t * dt) for t in range(n_steps)}
time_graph = {t: [(t + 1) % n_steps] for t in range(n_steps)}

vx_grad = gradient(path_x, time_graph)
vx = {t: vx_grad[t][(t + 1) % n_steps] for t in range(n_steps)}
print('Velocity vx:', vx)

## 2. Non-Holonomic SO(3) Rotation Matrix Composition (dot)

We integrate $SO(3)$ rotation matrix compositions $R_{k+1} = R_k \cdot dR_k$.

In [ ]:
current_r = {0: {0: 1.0, 1: 0.0, 2: 0.0}, 1: {0: 0.0, 1: 1.0, 2: 0.0}, 2: {0: 0.0, 1: 0.0, 2: 1.0}}

for t in range(n_steps):
    dr = {
        0: {0: 1.0, 1: -vx[t] * 0.05, 2: vx[t] * 0.05},
        1: {0: vx[t] * 0.05, 1: 1.0, 2: 0.0},
        2: {0: -vx[t] * 0.05, 1: 0.0, 2: 1.0},
    }
    current_r = dot(current_r, dr)

print('Final SO(3) Rotation Matrix R(T):', current_r)

## 3. Spatial Path Tracking & Sparsity Audit (sparsity)

We evaluate Euclidean trajectory tracking deviations and matrix sparsity.

In [ ]:
total_dev = sum(math.dist((path_x[t], path_y[t]), (path_x[t] * 0.9, path_y[t] * 0.9)) for t in range(n_steps))
print(f'Total Path Tracking Deviation: {total_dev:.4f} units')
print(f'SO(3) Matrix Sparsity:        {sparsity(current_r) * 100:.1f}%')